# Decode and rename data

`data/processed/dua_<start>-<end>.parquet` stores one row per declaration item, with every column still named and coded exactly as DNA publishes it (`DD*` tags from the XML spec). This step does two things, both written to a brand-new file (`..._decoded.parquet`) so the source parquet from 0.1 is never touched:

1. **Decode**: resolve the codes we can verify against an official catalog into a human-readable description, overwriting the code in place. A code with no verified match is left as its original raw value (not turned into a description, not nulled out).
2. **Rename**: give every column an intuitive name instead of DNA's internal tag. Two naming choices (`TIPO_REGIMEN`/`REGIMEN_ADUANERO` and `PAIS_PROCEDENCIA_DESTINO`) had several reasonable options and were picked by the project owner (see notes in the table below). `DDUNID_TRANS` is left with its raw name, since its meaning isn't documented anywhere found.

## What DNA confirms exists, and what it actually publishes

Uruguay's Dirección Nacional de Aduanas (DNA) publishes several official specs for the DUA message that name an internal lookup table for every coded field below ("Tabla de Aduanas", "Tabla de Regímenes", "Tabla Países", "Tabla Modo Transporte en frontera", etc.), confirming the codes are DNA-maintained and not arbitrary. But DNA has not published most of those tables as standalone public catalogs — they're only exposed through dropdowns in its LUCIA declaration system, not as a downloadable CSV/PDF. Guessing at those mappings would silently produce wrong descriptions, so **only three catalogs are decoded**; every other coded column is left as DNA's raw code (just renamed).

Sources consulted (Dirección Nacional de Aduanas, https://www.aduanas.gub.uy):

- "DUA de Importación" (field spec): https://www.aduanas.gub.uy/innovaportal/file/14492/1/dua-de-importacion.pdf
- "Instructivo de llenado del DUA para Exportación": https://www.aduanas.gub.uy/innovaportal/file/14467/1/anexo11_r_g_73.pdf
- "Procedimiento DUA Digital - Importación": https://www.aduanas.gub.uy/innovaportal/file/9902/1/dua_digital__importacion_anexo.pdf
- "Anexo II: Códigos de documentos obligatorios del Certificado de Origen Digital": https://www.aduanas.gub.uy/innovaportal/file/25234/1/anexo-ii-codigos.pdf
- Field descriptions for the rename column: `docs/FormatoDUADiariosPublicos.htm` (DNA's own DUA XML spec)

| Raw column | New name | Decoded? | Notes |
|---|---|---|---|
| `DDANO_PRESE` | `ANIO_PRESENTACION` | No | Year the DUA was filed |
| `DDNUME_CORRE_PUBLICO` | `NUMERO_PUBLICO_DUA` | No | Repeats every year; combine with `ANIO_PRESENTACION` to identify a declaration (see 0.1) |
| `DDFECH_INGSI` | `FECHA_DUA` | No | |
| `DDTIPO_REGI` | `TIPO_REGIMEN` | Yes (I/E/T → Importación/Exportación/Tránsito) | Kept distinct from `REGIMEN_ADUANERO` (`DDCODI_REGI`) — different concepts, both called "régimen" in Spanish; project owner's choice |
| `DDPAIS_ORIGE` | `PAIS_ORIGEN` | No — no published country-code catalog found (checked ALADI too) | |
| `DDPART_NANDI` | `NCM` | No — 10-digit tariff code, too large to embed. Official lookup: [NCM classification](https://www.aduanas.gub.uy/innovaportal/v/8495/3/innova.front/ncm-.html), [Arancel Nacional](https://www.gub.uy/ministerio-economia-finanzas/politicas-y-gestion/nomenclatura-aranceles-uruguay) (MEF) | |
| `DDPUER_EMBAR` | `PAIS_PROCEDENCIA_DESTINO` | No — same unpublished "Tabla Países" | País de procedencia for imports, país destino for exports — one field, dual meaning; project owner's choice |
| `DDCONV_INTER` | `CONVENIO_INTERNACIONAL` | No — DNA confirms an "Acuerdos" table exists; only per-agreement *document* codes were found (Anexo II above), not the agreement code itself | |
| `DDCODI_LIBER` | `CODIGO_EXONERACION` | No — table exists, no published values found | |
| `DDQUNICOM` | `CANTIDAD_UNIDADES_COMERCIALES` | No | |
| `DDTUNICOM` | `TIPO_UNIDAD_COMERCIAL` | No — table exists, no published values found | |
| `DDUNID_FIQTY` | `CANTIDAD_UNIDADES_FISICAS` | No | |
| `DDUNID_FIDES` | `TIPO_UNIDAD_FISICA` | No — defined per-NCM-item in the Arancel Nacional itself, no standalone catalog | |
| `DDVAD_INCR` | `VALOR_ADUANA` | No | |
| `DDIMA_DOLAR` | `IMADUNI_USD` | No | Acronym kept as-is; no verified source for its full expansion |
| `DDLIMA_DOLAR` | `IMADUNI_LIBERADO_USD` | No | |
| `DDRMI_DOLAR` | `RECARGO_MINIMO_USD` | No | |
| `DDLRMI_DOLAR` | `RECARGO_MINIMO_LIBERADO_USD` | No | |
| `DDRAD_DOLAR` | `RECARGO_ADICIONAL_USD` | No | |
| `DDLRAD_DOLAR` | `RECARGO_ADICIONAL_LIBERADO_USD` | No | |
| `DDRMO_DOLAR` | `RECARGO_MOVIL_USD` | No | |
| `DDLRMO_DOLAR` | `RECARGO_MOVIL_LIBERADO_USD` | No | |
| `DDIVA_DOLAR` | `IVA_USD` | No | |
| `DDLIVA_DOLAR` | `IVA_LIBERADO_USD` | No | |
| `DDLIVAA_DOLA` | `IVA_ANTICIPO_LIBERADO_USD` | No | |
| `DDIVAA_DOLAR` | `IVA_ANTICIPO_USD` | No | |
| `DDPOR_IMADUN` | `IMADUNI_PORCENTAJE` | No | |
| `DDPOR_RMI` | `RECARGO_MINIMO_PORCENTAJE` | No | |
| `DDPOR_RAD` | `RECARGO_ADICIONAL_PORCENTAJE` | No | |
| `DDPOR_IVA` | `IVA_PORCENTAJE` | No | |
| `DDPOR_IVAA` | `IVA_ANTICIPO_PORCENTAJE` | No | |
| `DDVIA_TRANSP` | `VIA_TRANSPORTE` | Partial — only code `A` confirmed (mercadería que cruza por sus propios medios); codes `1`-`9`, `B` undocumented | |
| `DDUNID_TRANS` | *(unchanged)* | No | Not in DNA's official field spec at all (only published 2016–2019); meaning unverified anywhere found, so left as the raw DNA tag rather than guessed at |
| `DDADUAINGEGR` | `ADUANA_INGRESO_EGRESO` | Partial — only Montevideo (`001`), Carrasco (`002`) and Aceguá (`009`) confirmed; ~15 other office codes present but unverified | |
| `DDTIPO_DOCUM` | `TIPO_DOCUMENTO_IMPOEXPO` | Yes for codes 2, 4, 6 (verbatim from two independent DNA instructivos); other codes (e.g. `9`) left as-is | |
| `DDLIBR_TRIBU` | `NUMERO_DOCUMENTO_IMPOEXPO` | No | The importer/exporter's actual document number (RUT, CI, etc.), paired with `TIPO_DOCUMENTO_IMPOEXPO` |
| `DDPESO_BRUTO` | `PESO_BRUTO` | No | |
| `DDPESO_NETO` | `PESO_NETO` | No | |
| `DDDNOMBRE` | `NOMBRE_IMPORTADOR_EXPORTADOR` | No | |
| `DDCODI_REGI` | `REGIMEN_ADUANERO` | No — table exists, no published values found | Kept distinct from `TIPO_REGIMEN` (`DDTIPO_REGI`); project owner's choice |
| `DDTIPO_OPERA` | `SUBREGIMEN` | No — table exists, no published values found | |
| `FUENTE_PERIODO`, `FUENTE_TIPO` | *(unchanged)* | — | Added by 0.1's own pipeline, not a DNA field — already intuitive |

If the missing catalogs matter for the analysis, the DUA XML spec's language ("tabla correspondiente") implies DNA holds them internally and could be asked for them directly (info@aduanas.gub.uy). DNA's own "Consultas DUA" statistics tool (https://www.aduanas.gub.uy/innovaportal/v/18714/6/innova.front/consultas-dua.html) also lets you filter by these same fields interactively, which is a way to cross-check any code by eye without a full catalog.

Numeric columns are already typed from 0.1 (`Float64`/`Date`/`Int16`), so no casting happens here.


In [ ]:
import logging
import re

import polars as pl
from uruguay_dua_comex import utils

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

data_processed = utils.data_processed_dir()

# 0.1 names the final table dua_<start>-<end>.parquet; the year range depends
# on which raw files were available when it was built, so it's found by
# pattern instead of a hardcoded name. This matches only that exact name --
# not dua_anulados_*.parquet, and not this notebook's own past output,
# dua_<start>-<end>_decoded.parquet.
SOURCE_PATTERN = re.compile(r"^dua_\d{4}-\d{4}\.parquet$")
source_candidates = [p for p in data_processed.glob("dua_*.parquet") if SOURCE_PATTERN.match(p.name)]
if len(source_candidates) != 1:
    raise FileNotFoundError(
        f"Expected exactly one dua_<start>-<end>.parquet file in {data_processed}, found {source_candidates}."
    )
source_path = source_candidates[0]
output_path = source_path.with_name(source_path.stem + "_decoded.parquet")

# Official code catalogs
# -----------------------
# Only codes verified against an official DNA source are mapped here; see the
# markdown cell above for what was checked and why every other coded column
# (DDPAIS_ORIGE, DDPUER_EMBAR, DDPART_NANDI, DDCONV_INTER, DDCODI_LIBER,
# DDTUNICOM, DDUNID_FIDES, DDVIA_TRANSP, DDCODI_REGI, DDTIPO_OPERA) is left
# as DNA's raw code instead of guessed at.

# DDTIPO_REGI: <DDTIPO_REGI> "Tipo Régimen (Importación/Exportación)" in
# docs/FormatoDUADiariosPublicos.htm. T for Tránsito is not named there but is
# the only regime code besides I/E that appears in the data (see 0.1).
TIPO_REGI = {"I": "Importación", "E": "Exportación", "T": "Tránsito"}

# DDTIPO_DOCUM: verbatim from DNA's "DUA de Importación" and "Instructivo de
# llenado del DUA para Exportación" field specs (see markdown cell for URLs).
TIPO_DOCUM = {
    "2": "Cédula de identidad de la R.O.U.",
    "4": "R.U.T.",
    "6": "Pasaporte, para extranjeros",
}

# DDADUAINGEGR: partial -- only these three offices are named in the DNA PDFs
# above; every other office code in the data is real but unverified.
ADUANAS = {
    "001": "Montevideo",
    "002": "Carrasco",
    "009": "Aceguá",
}

# Codes are decoded in place, under their raw DD* name, before the rename
# below -- so this dict has to use the raw names, not the new ones.
DECODED_COLUMNS = {
    "DDTIPO_REGI": TIPO_REGI,
    "DDTIPO_DOCUM": TIPO_DOCUM,
    "DDADUAINGEGR": ADUANAS,
}

# Intuitive names for DNA's raw DD* tags; see the markdown cell above for the
# full column-by-column rationale, sources and the two naming decisions
# (TIPO_REGIMEN/REGIMEN_ADUANERO, PAIS_PROCEDENCIA_DESTINO) made by the
# project owner. DDUNID_TRANS is deliberately left out: its meaning isn't
# documented anywhere found, so it keeps its raw DNA tag instead of a guess.
RENAME_COLUMNS = {
    "DDANO_PRESE": "ANIO_PRESENTACION",
    "DDNUME_CORRE_PUBLICO": "NUMERO_PUBLICO_DUA",
    "DDFECH_INGSI": "FECHA_DUA",
    "DDTIPO_REGI": "TIPO_REGIMEN",
    "DDPAIS_ORIGE": "PAIS_ORIGEN",
    "DDPART_NANDI": "NCM",
    "DDPUER_EMBAR": "PAIS_PROCEDENCIA_DESTINO",
    "DDCONV_INTER": "CONVENIO_INTERNACIONAL",
    "DDCODI_LIBER": "CODIGO_EXONERACION",
    "DDQUNICOM": "CANTIDAD_UNIDADES_COMERCIALES",
    "DDTUNICOM": "TIPO_UNIDAD_COMERCIAL",
    "DDUNID_FIQTY": "CANTIDAD_UNIDADES_FISICAS",
    "DDUNID_FIDES": "TIPO_UNIDAD_FISICA",
    "DDVAD_INCR": "VALOR_ADUANA",
    "DDIMA_DOLAR": "IMADUNI_USD",
    "DDLIMA_DOLAR": "IMADUNI_LIBERADO_USD",
    "DDRMI_DOLAR": "RECARGO_MINIMO_USD",
    "DDLRMI_DOLAR": "RECARGO_MINIMO_LIBERADO_USD",
    "DDRAD_DOLAR": "RECARGO_ADICIONAL_USD",
    "DDLRAD_DOLAR": "RECARGO_ADICIONAL_LIBERADO_USD",
    "DDRMO_DOLAR": "RECARGO_MOVIL_USD",
    "DDLRMO_DOLAR": "RECARGO_MOVIL_LIBERADO_USD",
    "DDIVA_DOLAR": "IVA_USD",
    "DDLIVA_DOLAR": "IVA_LIBERADO_USD",
    "DDLIVAA_DOLA": "IVA_ANTICIPO_LIBERADO_USD",
    "DDIVAA_DOLAR": "IVA_ANTICIPO_USD",
    "DDPOR_IMADUN": "IMADUNI_PORCENTAJE",
    "DDPOR_RMI": "RECARGO_MINIMO_PORCENTAJE",
    "DDPOR_RAD": "RECARGO_ADICIONAL_PORCENTAJE",
    "DDPOR_IVA": "IVA_PORCENTAJE",
    "DDPOR_IVAA": "IVA_ANTICIPO_PORCENTAJE",
    "DDVIA_TRANSP": "VIA_TRANSPORTE",
    "DDADUAINGEGR": "ADUANA_INGRESO_EGRESO",
    "DDTIPO_DOCUM": "TIPO_DOCUMENTO_IMPOEXPO",
    "DDLIBR_TRIBU": "NUMERO_DOCUMENTO_IMPOEXPO",
    "DDPESO_BRUTO": "PESO_BRUTO",
    "DDPESO_NETO": "PESO_NETO",
    "DDDNOMBRE": "NOMBRE_IMPORTADOR_EXPORTADOR",
    "DDCODI_REGI": "REGIMEN_ADUANERO",
    "DDTIPO_OPERA": "SUBREGIMEN",
}

lf = pl.scan_parquet(source_path)

# Coverage has to be measured against the raw codes before overwriting them:
# once a column is decoded in place, a raw code left untouched (no catalog
# match) looks the same as it did before, so this is the only point where
# "was this one decoded?" can still be told apart. is_in(...) is null for a
# null code, and fill_null(True) counts a null code as undecoded too.
coverage = lf.select(
    [
        (~pl.col(column).is_in(list(catalog))).fill_null(True).sum().alias(f"{column}_sin_decodificar")
        for column, catalog in DECODED_COLUMNS.items()
    ]
    + [pl.len().alias("total_filas")]
).collect()
logger.info(coverage)

# Overwriting the code in place is safe here: this writes a brand-new file,
# the source parquet from 0.1 is untouched. A code with no catalog match
# (undocumented, or not yet verified) passes through unchanged rather than
# becoming a wrong description or null.
for column, catalog in DECODED_COLUMNS.items():
    lf = lf.with_columns(pl.col(column).replace(catalog))

lf = lf.rename(RENAME_COLUMNS)

logger.info(f"Writing {output_path}...")
lf.sink_parquet(output_path)

# Preview from the written output rather than re-running the decode pipeline: reading
# back from disk only re-scans the already-decoded, compressed result.
decoded = pl.scan_parquet(output_path)
preview_columns = [RENAME_COLUMNS.get(column, column) for column in DECODED_COLUMNS]
print(decoded.select(preview_columns).head().collect())

logger.info(f"Wrote decoded dataset to {output_path}")